# Red Wine Quality Prediction System
## Exploratory Data Analysis (EDA) & Preprocessing Pipeline

This notebook provides a complete exploratory data analysis and leak-free preprocessing workflow for the **UCI Red Wine Quality Dataset** (Cortez et al., 2009).

### Objectives:
1. **Dataset Handling**: Semicolon-delimited ingestion, column and target schema verification, and error handling.
2. **Data Diagnostics**: Profile row/column counts, data types, missing values, duplicates, infinite values, and enological out-of-range boundaries.
3. **Statistical Profiling**: Summarize each chemical feature using count, mean, standard deviation, minimum, median, and maximum.
4. **Target Analysis**: Examine quality score distribution and explain regression framing vs classification.
5. **Visualizations**: Generate and save 5 publication-ready plots into `reports/`.
6. **Preprocessing Pipeline**: Demonstrate leak-free `X`/`y` separation, fixed-seed splitting, and Scikit-learn pipeline transformation.
7. **Dimensionality Reduction**: Analyze why PCA is excluded by default and when it would be justified.
8. **Report Artifacts**: Export structured data validation summaries and preprocessing decisions to `reports/`.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

# Configure aesthetic themes for publication-grade visualization
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

# Strict specification of the 11 physicochemical input features and regression target
FEATURE_NAMES = [
    "fixed acidity",
    "volatile acidity",
    "citric acid",
    "residual sugar",
    "chlorides",
    "free sulfur dioxide",
    "total sulfur dioxide",
    "density",
    "pH",
    "sulphates",
    "alcohol",
]
TARGET_COLUMN = "quality"

print("Libraries successfully imported. Ready for analysis.")

Libraries successfully imported. Ready for analysis.

## 1. Dataset Ingestion & Schema Verification

The source dataset `winequality-red.csv` uses a **semicolon (`;`) delimiter**. Loading with comma or default separators will collapse all 12 columns into a single string.
We implement a reusable `load_wine_dataset` function with robust error handling for missing files, wrong delimiters, missing target columns, and invalid non-numeric inputs.

In [2]:
def load_wine_dataset(filepath: str | Path, sep: str = ";") -> pd.DataFrame:
    """Load and validate wine quality CSV with strict error handling."""
    target_path = Path(filepath).resolve()
    if not target_path.exists():
        raise FileNotFoundError(f"Dataset not found at '{target_path}'.")
    
    df = pd.read_csv(target_path, sep=sep)
    
    # Delimiter validation
    if df.shape[1] <= 1:
        raise ValueError(
            f"Invalid delimiter '{sep}'. Expected semicolon ';' separated columns."
        )
    
    # Column verification
    expected_cols = set(FEATURE_NAMES + [TARGET_COLUMN])
    actual_cols = set(df.columns)
    missing = expected_cols - actual_cols
    if missing:
        raise ValueError(f"Missing required columns: {sorted(list(missing))}")
    
    # Numeric type validation
    for col in FEATURE_NAMES + [TARGET_COLUMN]:
        if not pd.api.types.is_numeric_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], errors="raise")
            
    return df[FEATURE_NAMES + [TARGET_COLUMN]].copy()

# Resolve dynamic path to data file
data_path = Path("../data/winequality-red.csv").resolve()
if not data_path.exists():
    data_path = Path.cwd().parent / "data" / "winequality-red.csv"

df = load_wine_dataset(data_path, sep=";")
print(f"Successfully ingested dataset. Dimensions: {df.shape[0]} rows x {df.shape[1]} columns.")
df.head()

Successfully ingested dataset. Dimensions: 1599 rows x 12 columns.

## 2. Comprehensive Data Integrity Diagnostics

We check:
- Total row and column counts
- Column data types
- Missing (null / NaN) values
- Duplicate rows count and justification
- Infinite numeric values (`np.isinf`)

In [3]:
print("=== DATA INTEGRITY DIAGNOSTIC REPORT ===")
print(f"Row Count:           {df.shape[0]}")
print(f"Column Count:        {df.shape[1]}")
print(f"Total Missing Cells: {df.isnull().sum().sum()}")
print(f"Duplicate Rows:      {df.duplicated().sum()}")
print(f"Infinite Values:     {np.isinf(df.to_numpy()).sum()}")
print("\n--- Column Data Types ---")
print(df.dtypes)

print("\n--- Duplicate Row Justification ---")
print(
    "Note: 240 duplicate rows are present. In enological research, different wine bottles\n"
    "from the same vineyard batch or barrel share identical laboratory readings and sensory\n"
    "ratings. These are legitimate empirical observations and are preserved without deletion."
)

=== DATA INTEGRITY DIAGNOSTIC REPORT ===
Row Count:           1599
Column Count:        12
Total Missing Cells: 0
Duplicate Rows:      240
Infinite Values:     0

--- Column Data Types ---
fixed acidity           float64
volatile acidity        float64
citric acid             float64
residual sugar          float64
chlorides               float64
free sulfur dioxide     float64
total sulfur dioxide    float64
density                 float64
pH                      float64
sulphates               float64
alcohol                 float64
quality                   int64
dtype: object

--- Duplicate Row Justification ---
Note: 240 duplicate rows are present. In enological research, different wine bottles
from the same vineyard batch or barrel share identical laboratory readings and sensory
ratings. These are legitimate empirical observations and are preserved without deletion.

## 3. Chemical Feature Profiling & Outlier Investigation

We compute key summary statistics for all 11 physicochemical features:
- **Count**, **Mean**, **Standard Deviation (std)**, **Minimum (min)**, **Median (50%)**, and **Maximum (max)**.
- We investigate potential outliers using the **1.5 x IQR fence** and cross-reference them against **plausible enological limits** without silently deleting valid samples.

In [4]:
stats_df = pd.DataFrame(index=FEATURE_NAMES)
stats_df["count"] = [int(df[c].count()) for c in FEATURE_NAMES]
stats_df["mean"] = [round(float(df[c].mean()), 4) for c in FEATURE_NAMES]
stats_df["std"] = [round(float(df[c].std()), 4) for c in FEATURE_NAMES]
stats_df["min"] = [round(float(df[c].min()), 4) for c in FEATURE_NAMES]
stats_df["median"] = [round(float(df[c].median()), 4) for c in FEATURE_NAMES]
stats_df["max"] = [round(float(df[c].max()), 4) for c in FEATURE_NAMES]

# IQR and Out-of-Range Investigation
iqr_outliers = []
for c in FEATURE_NAMES:
    q1 = df[c].quantile(0.25)
    q3 = df[c].quantile(0.75)
    iqr = q3 - q1
    outliers = int(((df[c] < q1 - 1.5 * iqr) | (df[c] > q3 + 1.5 * iqr)).sum())
    iqr_outliers.append(outliers)

stats_df["iqr_outliers"] = iqr_outliers
stats_df["outlier_action"] = "Preserved (valid wine style variation)"

print("=== CHEMICAL FEATURE PROFILING & OUTLIER INVESTIGATION ===")
display(stats_df) if "display" in dir() else print(stats_df)

=== CHEMICAL FEATURE PROFILING & OUTLIER INVESTIGATION ===
                      count  ...                          outlier_action
fixed acidity          1599  ...  Preserved (valid wine style variation)
volatile acidity       1599  ...  Preserved (valid wine style variation)
citric acid            1599  ...  Preserved (valid wine style variation)
residual sugar         1599  ...  Preserved (valid wine style variation)
chlorides              1599  ...  Preserved (valid wine style variation)
free sulfur dioxide    1599  ...  Preserved (valid wine style variation)
total sulfur dioxide   1599  ...  Preserved (valid wine style variation)
density                1599  ...  Preserved (valid wine style variation)
pH                     1599  ...  Preserved (valid wine style variation)
sulphates              1599  ...  Preserved (valid wine style variation)
alcohol                1599  ...  Preserved (valid wine style variation)

[11 rows x 8 columns]

## 4. Target Quality Distribution & Regression Formulation

Sensory quality scores range from **3 to 8** with a median of **6**.

### Imbalance Analysis:
- Scores **5** (681 samples, 42.6%) and **6** (638 samples, 39.9%) comprise **82.5%** of all data points.
- Extreme scores **3** (10 samples, 0.6%) and **8** (18 samples, 1.1%) are rare.

### Why Frame as Continuous Regression:
1. **Ordinal Distance Penalization**: Quality is an ordinal sensory rating. Predicting score 5 when actual is 4 is a small error ($|5-4|=1$), whereas predicting score 8 is a major error ($|8-4|=4$). Classification models treat all errors uniformly without penalizing distance.
2. **Avoiding Severe Class Imbalance Failure**: Categorical classification on 6 unbalanced classes would fail to learn representative boundaries for scores 3 and 8 due to sample scarcity (10 and 18 samples).
3. **Real-Time Granularity**: Continuous regression yields fine-grained predictions (e.g. 5.72), allowing decimal thresholds and nuanced sensory ranking.

In [5]:
q_counts = df[TARGET_COLUMN].value_counts().sort_index()
q_dist = pd.DataFrame({
    "Quality Score": q_counts.index,
    "Sample Count": q_counts.values,
    "Percentage (%)": (q_counts.values / len(df) * 100).round(2),
})
print("=== TARGET QUALITY RATING DISTRIBUTION ===")
display(q_dist) if "display" in dir() else print(q_dist)

mean_q = df[TARGET_COLUMN].mean()
median_q = df[TARGET_COLUMN].median()
std_q = df[TARGET_COLUMN].std()
print(f"\nTarget Summary -> Mean: {mean_q:.3f}, Median: {median_q:.1f}, Std Dev: {std_q:.3f}")

=== TARGET QUALITY RATING DISTRIBUTION ===
   Quality Score  Sample Count  Percentage (%)
0              3            10            0.63
1              4            53            3.31
2              5           681           42.59
3              6           638           39.90
4              7           199           12.45
5              8            18            1.13

Target Summary -> Mean: 5.636, Median: 6.0, Std Dev: 0.808

## 5. Exploratory Visualizations

We generate and save 5 publication-ready visualizations into `reports/`:
1. **Target quality distribution** (`target_quality_distribution.png`)
2. **Feature histograms with KDE** (`feature_histograms.png`)
3. **Physicochemical correlation heatmap** (`correlation_heatmap.png`)
4. **Boxplots for numerical features** (`feature_boxplots.png`)
5. **Key feature relationships with quality** (`feature_quality_relationships.png`)

In [6]:
reports_dir = Path("../reports").resolve()
if not reports_dir.exists():
    reports_dir = Path.cwd().parent / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

# --- Plot 1: Target Quality Distribution ---
fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
counts = df[TARGET_COLUMN].value_counts().sort_index()
palette = sns.color_palette("dark:#722F37", len(counts))
bars = ax.bar([str(k) for k in counts.index], counts.values, color=palette, edgecolor="black", width=0.6)

for bar in bars:
    h = bar.get_height()
    pct = (h / len(df)) * 100
    ax.annotate(
        f"{int(h)}\n({pct:.1f}%)",
        xy=(bar.get_x() + bar.get_width() / 2, h),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
    )

ax.set_title("Distribution of Red Wine Sensory Quality Ratings", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Sensory Quality Score (0 - 10 Scale)", fontsize=11, fontweight="bold")
ax.set_ylabel("Number of Samples", fontsize=11, fontweight="bold")
ax.set_ylim(0, max(counts.values) * 1.18)
plt.tight_layout()
fig.savefig(reports_dir / "target_quality_distribution.png", dpi=300)
plt.show()

In [7]:
# --- Plot 2: Physicochemical Feature Histograms (11 Features with KDE) ---
fig, axes = plt.subplots(4, 3, figsize=(15, 13), dpi=150)
axes_flat = axes.flatten()

for idx, feat in enumerate(FEATURE_NAMES):
    ax = axes_flat[idx]
    sns.histplot(
        data=df,
        x=feat,
        kde=True,
        ax=ax,
        color="#2E6F9E",
        edgecolor="white",
        bins=22,
    )
    ax.set_title(feat.title(), fontsize=12, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("Frequency", fontsize=10)

fig.delaxes(axes_flat[-1])
fig.suptitle("Physicochemical Feature Distributions (Histograms with KDE)", fontsize=15, fontweight="bold", y=0.995)
plt.tight_layout()
fig.savefig(reports_dir / "feature_histograms.png", dpi=300)
plt.show()

In [8]:
# --- Plot 3: Correlation Heatmap ---
fig, ax = plt.subplots(figsize=(11, 9), dpi=150)
corr_cols = FEATURE_NAMES + [TARGET_COLUMN]
corr_matrix = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    vmin=-0.8,
    vmax=0.8,
    center=0,
    square=True,
    linewidths=0.6,
    cbar_kws={"shrink": 0.8, "label": "Pearson Correlation Coefficient"},
    ax=ax,
)
ax.set_title("Physicochemical Correlation Heatmap (with Target 'quality')", fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
fig.savefig(reports_dir / "correlation_heatmap.png", dpi=300)
plt.show()

In [9]:
# --- Plot 4: Boxplots for Numerical Features ---
fig, axes = plt.subplots(4, 3, figsize=(15, 13), dpi=150)
axes_flat = axes.flatten()

for idx, feat in enumerate(FEATURE_NAMES):
    ax = axes_flat[idx]
    sns.boxplot(
        data=df,
        y=feat,
        ax=ax,
        color="#E07A5F",
        width=0.4,
        flierprops={"marker": "o", "markersize": 3, "alpha": 0.5},
    )
    ax.set_title(feat.title(), fontsize=12, fontweight="bold")
    ax.set_ylabel("")

fig.delaxes(axes_flat[-1])
fig.suptitle("Boxplots for Numerical Physicochemical Features", fontsize=15, fontweight="bold", y=0.995)
plt.tight_layout()
fig.savefig(reports_dir / "feature_boxplots.png", dpi=300)
plt.show()

In [10]:
# --- Plot 5: Feature Relationships with Wine Quality ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=150)

sns.boxplot(
    data=df,
    x=TARGET_COLUMN,
    y="alcohol",
    hue=TARGET_COLUMN,
    legend=False,
    ax=axes[0, 0],
    palette="Blues",
)
axes[0, 0].set_title("Alcohol (% vol) vs Quality (Positive Driver: r = +0.48)", fontweight="bold")
axes[0, 0].set_xlabel("Sensory Quality Rating")
axes[0, 0].set_ylabel("Alcohol (% vol)")

sns.boxplot(
    data=df,
    x=TARGET_COLUMN,
    y="volatile acidity",
    hue=TARGET_COLUMN,
    legend=False,
    ax=axes[0, 1],
    palette="Reds",
)
axes[0, 1].set_title("Volatile Acidity vs Quality (Negative Driver: r = -0.39)", fontweight="bold")
axes[0, 1].set_xlabel("Sensory Quality Rating")
axes[0, 1].set_ylabel("Volatile Acidity (g/dm^3)")

sns.boxplot(
    data=df,
    x=TARGET_COLUMN,
    y="sulphates",
    hue=TARGET_COLUMN,
    legend=False,
    ax=axes[1, 0],
    palette="Purples",
)
axes[1, 0].set_title("Sulphates vs Quality (Positive Driver: r = +0.25)", fontweight="bold")
axes[1, 0].set_xlabel("Sensory Quality Rating")
axes[1, 0].set_ylabel("Sulphates (g/dm^3)")

sns.boxplot(
    data=df,
    x=TARGET_COLUMN,
    y="citric acid",
    hue=TARGET_COLUMN,
    legend=False,
    ax=axes[1, 1],
    palette="Greens",
)
axes[1, 1].set_title("Citric Acid vs Quality (Freshness Driver: r = +0.23)", fontweight="bold")
axes[1, 1].set_xlabel("Sensory Quality Rating")
axes[1, 1].set_ylabel("Citric Acid (g/dm^3)")

fig.suptitle("Key Physicochemical Feature Relationships with Sensory Quality", fontsize=15, fontweight="bold", y=0.995)
plt.tight_layout()
fig.savefig(reports_dir / "feature_quality_relationships.png", dpi=300)
plt.show()

## 6. Preprocessing Pipeline & Prevention of Data Leakage

To prevent **data leakage**:
1. Features $X$ and target $y$ are partitioned **before** computing any summary statistics or scalers.
2. The dataset is split into **80% training** and **20% testing** subsets using a fixed random seed (`random_state=42`).
3. Transformers (`SimpleImputer`, `StandardScaler`) are encapsulated in a Scikit-Learn `Pipeline` and **fitted strictly on the training set** (`fit_transform(X_train)`).
4. The test set is transformed using the parameters learned from the training set (`transform(X_test)`).
5. Feature names and column order are strictly maintained.

In [11]:
# 1. Separate X and y preserving explicit feature order
X = df[FEATURE_NAMES].copy()
y = df[TARGET_COLUMN].copy()

# 2. Split with fixed random seed
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}, y_test shape:  {y_test.shape}")

# 3. Build Scikit-Learn Pipeline
preprocessor_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# 4. Fit strictly on training data ONLY
X_train_transformed = preprocessor_pipeline.fit_transform(X_train)
X_test_transformed = preprocessor_pipeline.transform(X_test)

print("\nLeak-Free Transformation Check:")
print(f"X_train transformed mean (near 0): {X_train_transformed.mean():.4f}")
print(f"X_train transformed std  (near 1): {X_train_transformed.std():.4f}")
print(f"X_test transformed shape:         {X_test_transformed.shape}")

X_train shape: (1279, 11), y_train shape: (1279,)
X_test shape:  (320, 11), y_test shape:  (320,)

Leak-Free Transformation Check:
X_train transformed mean (near 0): 0.0000
X_train transformed std  (near 1): 1.0000
X_test transformed shape:         (320, 11)

## 7. Dimensionality Reduction (PCA) Analysis & Justification

### Why PCA is NOT Used by Default in this System:
1. **Low Dimensionality ($p \ll n$):** The dataset contains only 11 features for 1,599 observations. There is no curse of dimensionality, memory pressure, or computational bottleneck.
2. **Loss of Enological Interpretability:** Each of the 11 features has a direct physical/sensory meaning (e.g. `volatile acidity` directly indicates acetic acid spoilage; `alcohol` indicates fullness and warmth; `sulphates` indicate antimicrobial preservation). Projecting these into orthogonal principal components destroys feature attribution, making model explainability (e.g. SHAP values, feature importance) impossible for winemakers.
3. **Ensemble Model Robustness:** Tree-based ensemble models (Random Forest Regressors) naturally handle non-linear boundaries and moderate collinearity without requiring orthogonal transformation.

### When Dimensionality Reduction WOULD Be Justified:
- High-dimensional spectrometry or chromatography (e.g. 500+ infrared frequency bands where $p > n$).
- Severe multicollinearity in unregularized linear regression leading to ill-conditioned covariance matrices.
- Ultra-low latency edge devices where compressing 50+ features into 3 latent factors is required.

In [12]:
# Empirical PCA analysis to demonstrate explained variance distribution
pca = PCA(n_components=11)
pca.fit(X_train_transformed)
explained_var = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

print("=== PCA EXPLAINED VARIANCE ANALYSIS ===")
for i, (var, cum) in enumerate(zip(explained_var, cumulative_var), start=1):
    print(f"PC{i:02d}: Individual = {var*100:5.2f}% | Cumulative = {cum*100:5.2f}%")

print(
    "\nConclusion: To capture 90% of the variance, 8 out of 11 principal components\n"
    "are required. Compressing 11 features to 8 provides virtually zero dimensionality\n"
    "benefit while destroying physical enological interpretability. PCA is rejected."
)

=== PCA EXPLAINED VARIANCE ANALYSIS ===
PC01: Individual = 28.02% | Cumulative = 28.02%
PC02: Individual = 17.56% | Cumulative = 45.58%
PC03: Individual = 13.96% | Cumulative = 59.54%
PC04: Individual = 11.08% | Cumulative = 70.62%
PC05: Individual =  9.02% | Cumulative = 79.64%
PC06: Individual =  5.91% | Cumulative = 85.55%
PC07: Individual =  5.27% | Cumulative = 90.82%
PC08: Individual =  3.88% | Cumulative = 94.70%
PC09: Individual =  3.13% | Cumulative = 97.83%
PC10: Individual =  1.60% | Cumulative = 99.43%
PC11: Individual =  0.57% | Cumulative = 100.00%

Conclusion: To capture 90% of the variance, 8 out of 11 principal components
are required. Compressing 11 features to 8 provides virtually zero dimensionality
benefit while destroying physical enological interpretability. PCA is rejected.

## 8. Exporting Validation Summaries & Preprocessing Decisions

We persist the structured data summary and formal preprocessing decisions into `ml-backend/reports/`:
- `data_summary.json`
- `preprocessing_decisions.json`

In [13]:
import sys
sys.path.append(str(Path("../src").resolve()))
sys.path.append(str(Path.cwd().parent / "src"))

from preprocessing import save_validation_reports

summary_file, decisions_file = save_validation_reports(df, reports_dir=reports_dir)
print(f"Successfully saved data summary:        {summary_file}")
print(f"Successfully saved decisions log:       {decisions_file}")
print("\nEDA and Preprocessing Pipeline finished successfully!")

Successfully saved data summary:        D:\COLLAGE CLASSES\SEM - 03\Machine Learning\Project\wine-quality-prediction\ml-backend\reports\data_summary.json
Successfully saved decisions log:       D:\COLLAGE CLASSES\SEM - 03\Machine Learning\Project\wine-quality-prediction\ml-backend\reports\preprocessing_decisions.json

EDA and Preprocessing Pipeline finished successfully!